In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["OPENCV_LOG_LEVEL"] = "ERROR"

In [3]:
from pathlib import Path

import pandas as pd
from jppype import Mosaic, vscode_theme
from tqdm import tqdm, trange

from fundus_toolkits import FundusData
from fundus_toolkits.utils.data_io import most_common_image_ext
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset, SampleInfo

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [23]:
DATASETS_ROOT = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/")
DATASETS_PATH = {
    dataset: DATASETS_ROOT / folder
    for dataset, folder in {
        "GAVE-train": "GAVE-train",
        "MAPLES-DR": "MAPLES-DR",
        "FundusAV": "Fundus-AV",
        "HRF": "HRF",
        "LES-AV": "LES-AV",
        "INSPIRE": "INSPIRE",
        "DRIVE_train": "AV_DRIVE/training",
        "DRIVE_test": "AV_DRIVE/test",
    }.items()
}
RAW = [path / "1-images" for path in DATASETS_PATH.values()]
TOPO = [path / "3-topo" for path in DATASETS_PATH.values()]
AV = [
    {
        "fvt": path / "2-av-pred_FVT",
        "automorph": path / "2-av-pred_Automorph",
        "gt": path / "2-av",
        "vascx": path / "2-av-pred_VascX",
    }
    for path in DATASETS_PATH.values()
]


## Preprocess Datasets


In [24]:
from fundus_vessels_toolkit.models import segment_av
from fundus_vessels_toolkit.models.wrappers.automorph import automorph_segment_av
from fundus_vessels_toolkit.models.wrappers.vascx import vascx_segment_av

for dataset_path in DATASETS_PATH.values():
    raw_path = dataset_path / "1-images"
    for raw_file in tqdm(Path(raw_path).glob(f"*{most_common_image_ext(raw_path)}")):
        fvt_out = dataset_path / "2-av-pred_FVT" / (raw_file.stem + ".png")
        automorph_out = dataset_path / "2-av-pred_Automorph" / (raw_file.stem + ".png")
        vascx_out = dataset_path / "2-av-pred_VascX" / (raw_file.stem + ".png")

        if fvt_out.exists() and automorph_out.exists() and vascx_out.exists():
            continue

        fundus = FundusData(image=raw_file)
        fundus_cropped, roi = fundus.crop_to_roi(return_roi=True)

        if not fvt_out.exists():
            segment_av(fundus_cropped)
            fundus.update(av=fundus_cropped.av, domain=roi).write_image(av=fvt_out)

        if not automorph_out.exists():
            automorph_segment_av(fundus_cropped)
            fundus.update(av=fundus_cropped.av, domain=roi).write_image(av=automorph_out, on_exists="overwrite")

        if not vascx_out.exists():
            vascx_segment_av(fundus_cropped)
            fundus.update(av=fundus_cropped.av, domain=roi).write_image(av=vascx_out)

50it [00:00, 7876.33it/s]
200it [00:00, 7451.24it/s]
100it [00:00, 9284.36it/s]
44it [00:00, 5948.98it/s]
22it [00:00, 4835.19it/s]
15it [00:00, 3489.63it/s]
20it [00:00, 4742.27it/s]
20it [00:00, 4071.35it/s]


In [25]:
dataset = BranchDigraphDataset.process_dirs(
    RAW,
    TOPO,
    AV,
    dataset_name=list(DATASETS_PATH.keys()),
    resize_to=1024,
    output_dir="tmp/ALL_DATA_V2",
    n_workers=0,
    mask_optic_disc=True,
)

Output()

Found 373 branch digraphs...


Output()

Processing...
Done!


## Define dataset splits


In [7]:
samples_by_dataset: dict[str, list[SampleInfo]] = {}
for sample in dataset.samples_info:
    samples_by_dataset.setdefault(sample.dataset, []).append(sample)

samples_stratification: dict[str, list[SampleInfo]] = {}


**Fundus AV**: stratify by pathology


In [8]:
for sample in samples_by_dataset["FundusAV"]:
    samples_stratification.setdefault(f"FundusAV-{sample.name[-1]}", []).append(sample)

**LES-AV**, **INSPIRE** and **GAVE-train**: no specific stratification


In [9]:
samples_stratification["LES-AV"] = samples_by_dataset["LES-AV"]
samples_stratification["INSPIRE"] = samples_by_dataset["INSPIRE"]
samples_stratification["GAVE-train"] = samples_by_dataset["GAVE-train"]
samples_stratification["HRF"] = samples_by_dataset["HRF"]

In [10]:
for samples in samples_stratification.values():
    N = len(samples)
    train_end, val_end = int(N * 0.7), int(N * 0.85)
    for sample in samples[:train_end]:
        sample.dataset_type = "train"
    for sample in samples[train_end:val_end]:
        sample.dataset_type = "validation"
    for sample in samples[val_end:]:
        sample.dataset_type = "test"

Use existing splits for **DRIVE** and **MAPLES-DR**


In [11]:
for sample in samples_by_dataset["DRIVE_train"]:
    sample.dataset_type = "validation" if sample.name.startswith(("31", "33", "35", "28")) else "train"
for sample in samples_by_dataset["DRIVE_test"]:
    sample.dataset_type = "test"

In [12]:
import maples_dr

maples_dr_test_samples_name = {s.name for s in maples_dr.load_test_set()}
maples_dr_test_samples: list[SampleInfo] = []
for sample in samples_by_dataset["MAPLES-DR"]:
    if sample.name in maples_dr_test_samples_name:
        maples_dr_test_samples.append(sample)
    else:
        sample.dataset_type = "train"
N_test = len(maples_dr_test_samples)
for sample in maples_dr_test_samples[: N_test // 2]:
    sample.dataset_type = "validation"
for sample in maples_dr_test_samples[N_test // 2 :]:
    sample.dataset_type = "test"

Thanks for using MAPLES-DR!
  When using this dataset in academic works,
  please cite: ]8;id=260030;https://doi.org/10.1038/s41597-024-03739-6\https://doi.org/10.1038/s41597-024-03739-6]8;;\

Check that all samples have a dataset type assigned and save the splits


In [13]:
samples_without_type = [
    sample for sample in dataset.samples_info if sample.dataset_type not in ("train", "validation", "test")
]
assert not samples_without_type, f"Samples without dataset type: {[s.name for s in samples_without_type]}"

dataset.save_manifest()

In [14]:
train_set, val_set, test_set = dataset.split_sets()


def count_dataset(dataset):
    counts = {}
    for sample in dataset.samples_info:
        counts[sample.dataset] = counts.get(sample.dataset, 0) + 1
    return counts


df = pd.DataFrame(
    {"train": count_dataset(train_set), "validation": count_dataset(val_set), "test": count_dataset(test_set)}
).T
df["TOTAL"] = df.sum(axis=1)
df

,FundusAV,HRF,LES-AV,MAPLES-DR,DRIVE_train,GAVE-train,INSPIRE,DRIVE_test,TOTAL
train,70.0,28.0,15.0,73.0,16.0,35.0,10.0,NaN,247.0
validation,15.0,6.0,3.0,16.0,4.0,7.0,2.0,NaN,53.0
test,15.0,7.0,4.0,16.0,NaN,8.0,3.0,20.0,73.0


## Bundle dataset


In [15]:
dataset.bundle("ALL_DATA_bundle_v2.tar.gz", overwrite=True)

In [4]:
bundled_dataset = BranchDigraphDataset("ALL_DATA_bundle_v2.tar.gz")

Processing...
Done!


In [6]:
train_set, val_set, test_set = bundled_dataset.split_sets()


def count_dataset(dataset):
    counts = {}
    for sample in dataset.samples_info:
        counts[sample.dataset] = counts.get(sample.dataset, 0) + 1
    return counts


df = pd.DataFrame(
    {"train": count_dataset(train_set), "validation": count_dataset(val_set), "test": count_dataset(test_set)}
)
df["all"] = df.sum(axis=1)
df = df.T
df["TOTAL"] = df.sum(axis=1)
df

,FundusAV,HRF,LES-AV,MAPLES-DR,DRIVE_train,GAVE-train,INSPIRE,DRIVE_test,TOTAL
train,70.0,28.0,15.0,73.0,16.0,35.0,10.0,NaN,247.0
validation,15.0,6.0,3.0,16.0,4.0,7.0,2.0,NaN,53.0
test,15.0,7.0,4.0,16.0,NaN,8.0,3.0,20.0,73.0
all,100.0,41.0,22.0,105.0,20.0,50.0,15.0,20.0,373.0


## Test


In [19]:
samples_src = BranchDigraphDataset.discover_paths(RAW, TOPO, AV, dataset_name=list(DATASETS_PATH.keys()))

In [20]:
from fundus_vessels_toolkit.utils.profiling import Profiler

with Profiler():
    sample_info = samples_src[200].process(
        output_dir=Path("tmp/test_data_process/"), resize_to=1024, mask_optic_disc=True, overwrite=True
    )
    sample = sample_info.load(load_av_maps=True, discard_gt_tree=False)

torch.Size([1, 3, 1992, 2656]) torch.Size([1, 3, 512, 512])
4


]8;id=778841;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:539\SampleSource.process]8;;\                                11.2s                                      (runs=1)
├── ]8;id=226645;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:541\Load fundus image]8;;\                               ↳2.1%  235.1ms                             (runs=1)
│   ├── ]8;id=158239;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:542\Read image from disk]8;;\                        0.86%    ↳ 41%   96.6ms                    (runs=1)
│   ├── ]8;id=802435;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:545\Crop to ROI & resize]8;;\                        1.15%    ↳ 55%  129.5ms                    (runs=1)
│   └── ]8;id=614985;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:561\Write processed fundus image]8;;\                0.40‰    ↳1.9%    4.6ms                    (runs=1)
├── ]8;id=271798;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:566\Load or compute OD and Macula]8;;\                   ↳ 21%     2.3s                             (runs=1)
│   └── ]8;id=154193;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:483\Segment OD and Macula]8;;\                         18%    ↳ 88%     2.0s                    (runs=1)
├── ]8;id=669992;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:579\Load and preprocess graphes]8;;\                     ↳ 72%     8.1s                             (runs=1)
│   ├── ]8;id=880483;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:618\Load AV map]8;;\                                 4.83%    ↳6.7%  543.2ms                    (runs=4, avg= 135.8ms)
│   │   ├── ]8;id=622921;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:619\FundusData.update]8;;\                       4.13%    5.73%    ↳ 85%  464.2ms           (runs=4, avg= 116.0ms)
│   │   ├── ]8;id=731795;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:622\remove OD from vessels]8;;\                  0.15%    0.21%    ↳3.1%   16.8ms           (runs=4, avg=   4.2ms)
│   │   └── ]8;id=870951;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:628\save AV map]8;;\                             0.55%    0.77%    ↳ 11%   62.1ms           (runs=4, avg=  15.5ms)
│   ├── ]8;id=724609;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:631\av2tree.to_vgraph]8;;\                             63%    ↳ 88%     7.1s                    (runs=4, avg=    1.8s)
│   │   ├── ]8;id=840347;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/pipelines/avseg_to_tree.py:400\Parse artery graph]8;;\                        25%      35%    ↳ 40%     2.8s           (runs=4, avg= 706.0ms)
│   │   │   ├── ]8;id=608703;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/pipelines/seg_to_graph.py:125\Skeletonize]8;;\                           20%      28%      32%    ↳ 80%     2.3s  (runs=4, avg= 563.8ms)
│   │   │   ├── ]8;id=560603;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/pipelines/seg_to_graph.py:127\Parse graph]8;;\                         0.59%    0.82%    0.93%    ↳2.3%   66.3ms  (runs=4, avg=  16.6ms)
│   │   │   ├── ]8;id=347907;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/pipelines/seg_to_graph.py:130\Simplify graph]8;;\    

In [21]:
from fundus_vessels_toolkit.pipelines.avseg_to_tree import GNNAVSegToTree


av2tree = GNNAVSegToTree()
fundus = sample.fundus.update(av=sample._av_maps["automorph"])

trees = av2tree(fundus)
# %timeit av2tree(fundus)

In [22]:
from fundus_vessels_toolkit.utils.jppype import draw_trees


v = fundus.draw()
draw_trees(trees, v, bspline_dir=True, interactive=True)
v

View2D()